# 실시간 날씨 데이터를 기반으로 충남·대전·세종 각 지역별 1시간 이내 교통사고확률 예측
- 팀명 : 국민대민쑤

## 코드 실행환경

In [ ]:
import platform
print(platform.platform())
!cat /etc/issue.net
!python --version
!nvidia-smi

## 구글 코랩 사용시 구글 드라이브 연결 사용

In [ ]:
# Set the local data directory before running the notebook.
DATA_PATH = "data/"


## Import & Install

In [ ]:
# 코랩 기준 필요 라이브러리 설치

!pip install catboost
!pip install haversine
!pip install optuna

In [ ]:
#Base & visualization
import pandas as pd
import random
import os
import numpy as np
import warnings
import matplotlib.pylab as plt
import seaborn as sns

#Feature engineering
import datetime
from haversine import haversine

#Scaling
from sklearn.preprocessing import StandardScaler

#Sklearn module & utils
from sklearn.model_selection import StratifiedKFold , KFold, train_test_split, cross_val_score, cross_validate

#Metric
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

#Modeling
import lightgbm as lgb
import catboost as cb
import optuna
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import cross_val_score

#Model save
import pickle

## Fix Seed

In [ ]:
#Seed 고정
class CFG:
    SEED = 42

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
seed_everything(CFG.SEED) # Seed 고정

## Data Load

In [ ]:
kp2020 = pd.read_csv(DATA_PATH + 'KP2020.csv', encoding = 'cp949')
kp2021 = pd.read_csv(DATA_PATH + 'KP2021.csv', encoding = 'cp949')
npa2020 = pd.read_csv(DATA_PATH + 'NPA2020.csv', encoding = 'cp949')
codeBook = pd.read_excel(DATA_PATH + 'codeBook_v3.xlsx')

In [ ]:
# 외부데이터(기상청)
temp2020 = pd.read_csv(DATA_PATH + '2020년기상청관측데이터.csv', encoding = 'cp949')
temp2021 = pd.read_csv(DATA_PATH + '2021년기상청관측데이터.csv', encoding = 'cp949')
temp2022 = pd.read_csv(DATA_PATH + '2022년기상청관측데이터.csv', encoding = 'cp949')
temp2023 = pd.read_csv(DATA_PATH + '2023년기상청관측데이터.csv', encoding = 'cp949')
temp2023_02 = pd.read_csv(DATA_PATH + '2023년최신기상청관측데이터.csv', encoding = 'cp949')
location = pd.read_csv(DATA_PATH + '관측지점정보.csv', encoding = 'cp949')

#### **외부데이터 설명**
- temp2020-temp2022 : 2020-2022년 날씨데이터
- temp2023 : 2023년 1월 18일까지 날씨데이터
- temp2023_02 : 20203년 1월 19일부터 2월 14일 09:00까지의 데이터
- location : 전국에 있는 기상관측소 위치 및위도,경도

[외부데이터 출처] : https://data.kma.go.kr/data/grnd/selectAsosRltmList.do?pgmNo=36

## EDA

In [ ]:
codeBook.head()

#### **- codeBook 정보**
**NPA_CL : 경찰청 구분**  
**EVT_STAT_CD : 사건상태코드**  
**EVT_CL_CD : 사건종별코드**  
**RPTER_SEX : 성별**

In [ ]:
# 교통사고 코드값 추출
codeBook.query("컬럼명 == 'EVT_CL_CD'").query("코드명 == '교통사고'")

In [ ]:
npa2020.sort_values(by='RECV_CPLT_DT')

In [ ]:
kp2020.sort_values(by='RECV_CPLT_DM')

In [ ]:
kp2021.sort_values(by='RECV_CPLT_DM')

- npa2020 : 2020년1월부터 2020년 11월까지의 데이터
- kp2020 : 2020년 12월 데이터
- kp2021 : 2021년1월부터 2023년1월18일까지인 데이터

#Feature engineering

### 날씨데이터

In [ ]:
# 2020년1월부터 2020년1월18일까지 병합
temp_all = pd.concat([temp2020,temp2021,temp2022,temp2023]).sort_values(by=["지점", "일시"]).reset_index(drop=True)

In [ ]:
temp_all

In [ ]:
#날씨데이터 결측치 확인
temp_all.isnull().sum()

In [ ]:
temp_all.지점.unique()

In [ ]:
지점_list = [129, 133, 177, 232, 235, 236, 238, 239]
temp_fine = pd.DataFrame(columns=['지점', '지점명', '일시', '기온(°C)', '풍속(m/s)', '풍향(16방위)', '습도(%)', '증기압(hPa)','이슬점온도(°C)', '현지기압(hPa)', '해면기압(hPa)', '전운량(10분위)', '시정(10m)','지면온도(°C)'])

In [ ]:
# 시계열데이터에서 일반적으로 결측치 처리에 좋은 interpolate(보간법)을 이용하여 결측치 처리
# interpolate 특성상 지역별로 결측치 처리를 안하면 다른 지역 데이터가 들어갈수 있으므로 지역별로 결측치 처리
for i in range(len(지점_list)):
  temp_fine = pd.concat([temp_fine,temp_all.query(f'지점=={지점_list[i]}').interpolate()])

In [ ]:
# 결측치 1547개 확인 -> 결측치 확인결과 세종시에서 발생
# 사유 : 세종시에서는 2020-03-05 이전까지 전운량측정을 안했음 -> 근처 대전시 값으로 대체
temp_fine.isnull().sum()

In [ ]:
# 세종시 미관측데이터 대전시 데이터로 보강
cloud_list = temp_fine.query('지점명=="대전"')['전운량(10분위)'][:1547].to_list()
temp_fine.loc[range(187152, 188699), '전운량(10분위)'] = cloud_list

In [ ]:
# 전운량데이터의 경우 10분위로 이루어져있는 범주형 피처
# interpolate때문에 소수점발생하므로 반올림진행
temp_fine["전운량(10분위)"] = temp_fine["전운량(10분위)"].round()

### 기상관측소 데이터

In [ ]:
# location데이터에서 충남,대전,세종에 있는 기상 관측소만 추출
location = location.query(f'지점 == {지점_list}')
location = location.drop_duplicates(subset='지점', keep='first').reset_index(drop=True)
location['위도'] = location['위도'].astype(float)

In [ ]:
location

### 교통사고 데이터

In [ ]:
# 교통사고 데이터만 추출
kp2020_traffic = kp2020.query('EVT_CL_CD == 401').reset_index(drop=True)
kp2021_traffic = kp2021.query('EVT_CL_CD == 401').reset_index(drop=True)
npa2020_traffic = npa2020.query('EVT_CL_CD == 401').reset_index(drop=True)

In [ ]:
# 교통사고 데이터중 위도,경도 결측치 제거
kp2020_traffic = kp2020_traffic.dropna(subset=['HPPN_X']).reset_index(drop=True)
kp2021_traffic = kp2021_traffic.dropna(subset=['HPPN_X']).reset_index(drop=True)
npa2020_traffic = npa2020_traffic.dropna(subset=['HPPN_X']).reset_index(drop=True)
npa2020_traffic = npa2020_traffic[npa2020_traffic.HPPN_X != 0]

In [ ]:
# npa2020,와 kp2020,kp2021 날짜 형식 통일
# 1시간 이내 데이터 예측이므로 ex) 00:00~00:59분까지 00:00으로 통일
def add_zeros(x):
    x = str(x)
    return x.zfill(6)

npa2020_traffic['RECV_CPLT_TM'] = npa2020_traffic['RECV_CPLT_TM'].apply(add_zeros)
npa2020_traffic['RECV_CPLT_DM'] = npa2020_traffic['RECV_CPLT_DT'].astype(str).str[:4] + '-' + npa2020_traffic['RECV_CPLT_DT'].astype(str).str[4:6] + '-' + npa2020_traffic['RECV_CPLT_DT'].astype(str).str[6:8] + ' ' + npa2020_traffic['RECV_CPLT_TM'].astype(str).str[:2]+':00'
npa2020_traffic = npa2020_traffic.drop(columns=['RECV_CPLT_DT','RECV_CPLT_TM','HPPN_OLD_ADDR'])

kp2020_traffic['RECV_CPLT_DM'] = '20' + kp2020_traffic['RECV_CPLT_DM']
kp2020_traffic['RECV_CPLT_DM'] = pd.to_datetime(kp2020_traffic['RECV_CPLT_DM'])
kp2020_traffic['RECV_CPLT_DM'] = kp2020_traffic['RECV_CPLT_DM'].dt.strftime('%Y-%m-%d %H')
kp2020_traffic['RECV_CPLT_DM'] = kp2020_traffic['RECV_CPLT_DM'] + ':00'

kp2021_traffic['RECV_CPLT_DM'] = '20' + kp2021_traffic['RECV_CPLT_DM']
kp2021_traffic['RECV_CPLT_DM'] = pd.to_datetime(kp2021_traffic['RECV_CPLT_DM'])
kp2021_traffic['RECV_CPLT_DM'] = kp2021_traffic['RECV_CPLT_DM'].dt.strftime('%Y-%m-%d %H')
kp2021_traffic['RECV_CPLT_DM'] = kp2021_traffic['RECV_CPLT_DM'] + ':00'

In [ ]:
# 교통사고데이터 병합
kp_all = pd.concat([kp2020_traffic,kp2021_traffic])
kp_all = kp_all.drop(columns=['RECV_DEPT_NM','HPPN_PNU_ADDR'])
traffic_all = pd.concat([npa2020_traffic,kp_all]).reset_index(drop=True)

In [ ]:
# haversine : 위도,경도 거리 계산 라이브러리(따로 설치필요)
# 교통사고데이터 위도,경도와 충남,대전,세종에 있는 모든 기상관측소 위도,경도 거리 계산 후 제일 가까운 관측소 위도,경도를 추출
traffic_all['지점'] = traffic_all.apply(lambda x: location.loc[np.argmin([haversine((x.HPPN_Y, x.HPPN_X), (loc.위도, loc.경도), unit='km') for i, loc in location.iterrows()]), '지점'], axis=1)

In [ ]:
# 지점별로 정렬 후 시간순으로 정렬
traffic_all = traffic_all[['지점','RECV_CPLT_DM']].sort_values(by=["지점", "RECV_CPLT_DM"]).reset_index(drop=True)

In [ ]:
# 교통사고 데이터에 '사고유무'컬럼 추가 후 날씨 데이터와 병합
# '지점'과'일시'를 기준으로 병합하여 날씨데이터 기준 1시간 간격 교통사고 유무를 파악할 수 있음
traffic_all['사고유무'] = 1
traffic_all.drop_duplicates(subset=['지점','RECV_CPLT_DM']).reset_index(drop=True)
traffic_all.rename(columns = {"RECV_CPLT_DM": "일시"}, inplace = True)
X = pd.merge(temp_fine, traffic_all, on=['지점', '일시'], how='left')
X['사고유무'] = X['사고유무'].fillna(0)
X['일시'] = pd.to_datetime(X['일시'])

In [ ]:
X

## 모델 생성 및 성능 확인을 위한 Train Test split

In [ ]:
# Predict확인을 위해 index값 저장
X_train_index = X.query('일시 <= "2022-01-18 22:00:00"').iloc[:,:3]
X_test_index = X.query('일시 > "2022-01-18 22:00:00"').iloc[:,:3]

# Train데이터 : 2020년1월1일 ~ 2022년 1월18일22시 데이터
# Test데이터 : 2022년1월18일23시 ~ 2023년1월18일23시 데이터
X_train = X.query('일시 <= "2022-01-18 22:00:00"').iloc[:,3:]
X_test = X.query('일시 > "2022-01-18 22:00:00"').iloc[:,3:]

# y_label 분리
y_train = X_train['사고유무']
X_train = X_train.drop(columns ='사고유무')
y_test = X_test['사고유무']
X_test = X_test.drop(columns ='사고유무')

In [ ]:
X_train.head()

#Scaling

In [ ]:
# 수치형 데이터 스케일링(전운량의 경우 범주형 속성에 가까우므로 제외)
num_features = X_train.drop(columns ='전운량(10분위)').columns

In [ ]:
# 일반적으로 분류에 성능이 좋은 StandardScaler 사용
scaler = StandardScaler()
X_train[num_features] = scaler.fit_transform(X_train[num_features])
X_test[num_features] = scaler.transform(X_test[num_features])

# Modeling

### LGBMClassifier

In [ ]:
# Define objective function for Optuna
def objective_lgbm(trial):
    # Define parameters to be optimized
    lgbm_params = {
        'objective': 'binary',
        'boosting_type': 'gbdt',
        'verbosity': -1,
        'random_state' : CFG.SEED,
        'num_leaves': trial.suggest_int('num_leaves', 2, 256),
        'learning_rate': trial.suggest_uniform('learning_rate', 0.01, 0.1),
        'feature_fraction': trial.suggest_uniform('feature_fraction', 0.1, 1.0),
        'bagging_fraction': trial.suggest_uniform('bagging_fraction', 0.1, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
    }
    
    # Create LGBM classifier with parameters
    lgbm_clf = lgb.LGBMClassifier(**lgbm_params)
    
    # Compute cross validation score
    lgbm_score = cross_val_score(lgbm_clf, X_train, y_train, cv=15).mean()
    
    return lgbm_score

# Run optuna to optimize hyperparameters
study_lgbm = optuna.create_study(direction='maximize')
study_lgbm.optimize(objective_lgbm, n_trials=15)

# Get best hyperparameters and create LGBM classifier with those parameters
lgbm_params = study_lgbm.best_params
lgbm_model = lgb.LGBMClassifier(**lgbm_params)

### CatBoostClassifier

In [ ]:
# Define objective function for Optuna for CatBoost
def objective_cb(trial):
    # Define parameters to be optimized
    cb_params = {
        'loss_function': 'Logloss',
        'iterations': 1000,
        'task_type' : "GPU",
        'verbose' : False,
        'random_state' : CFG.SEED,
        'learning_rate': trial.suggest_uniform('learning_rate', 0.01, 0.1),
        'depth': trial.suggest_int('depth', 2, 10),
        'l2_leaf_reg': trial.suggest_loguniform('l2_leaf_reg', 0.01, 10.0),
        'random_strength': trial.suggest_uniform('random_strength', 0.1, 1.0),
        'bagging_temperature': trial.suggest_uniform('bagging_temperature', 0.0, 10.0),
        'border_count': trial.suggest_int('border_count', 1, 255),
    }

    # Create Catboost classifier with parameters
    cb_clf = cb.CatBoostClassifier(**cb_params)
    
    # Compute cross validation score
    cb_score = cross_val_score(cb_clf, X_train, y_train, cv=5).mean()
    
    return cb_score    

# Run optuna to optimize hyperparameters
study_cb = optuna.create_study(direction='maximize')
study_cb.optimize(objective_cb, n_trials=5)

# Get best hyperparameters and create Catboost classifier with those parameters
cb_params = study_cb.best_params
catboost_model = cb.CatBoostClassifier(**cb_params)

### Ensemble

In [ ]:
# Create voting ensemble of LightGBM and CatBoost models
ensemble_model = VotingClassifier(estimators=[('lgbm', lgbm_model), ('catboost', catboost_model)], voting='soft')
ensemble_model.fit(X_train, y_train)

# y_pred는 사고유무(0,1)로 구성 y_pred_proba는 사고확률 %로 구성
y_pred = ensemble_model.predict(X_test)
y_pred_proba = ensemble_model.predict_proba(X_test)

In [ ]:
# 이전에 복사해두었던 index값 불러와서 submission 생성
X_test_y_t = X_test_index.copy()
X_test_y_p = X_test_index.copy()
X_test_y_proba = X_test_index.copy()
X_test_y_t['사고유무'] = y_test
X_test_y_p['사고유무'] = y_pred
X_test_y_proba['사고확률'] = [row[1] for row in y_pred_proba]
X_test_y_proba['사고확률'] = X_test_y_proba['사고확률'].apply(lambda x: str(round(x * 100, 2)) + '%')

## 학습 및 성능테스트 결과

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print('F1 Score : ', f1)
print('Accuracy : ', accuracy)

In [ ]:
X_test_y_p.head()

In [ ]:
X_test_y_proba.head()

#### 결과
- F1 Score :  0.8577479940831054
- Accuracy :  0.7910865343471248

#### **- 1시간 이내 교통사고발생을 5개를 예측하면 4개를 성공적으로 맞추는 준수한 성능을 가진 모델 개발하였고 사고확률까지 계산 가능하게 만들었음**



# **실사용 예측모델 생성(모든데이터 학습)**
- 주어진 데이터를 이용하여 실제 미래데이터 예측을 진행(2023년2월14일09:00까지 예측 진행)


#### Train 데이터 생성(2020년1월1일~2023년1월18일 데이터)

In [ ]:
# 모든 날짜가 들어있는 데이터 X에서 y_label 분리
y = X['사고유무']
X_index = X.iloc[:,:3]
X = X.drop(columns='사고유무')
X = X.iloc[:,3:]

#### Test 데이터 생성(2023년 1월19일~2023년2월14일09:00 데이터)

In [ ]:
# Test데이터 전처리
temp2023_02 = temp2023_02.sort_values(by=["지점", "일시"]).reset_index(drop=True)

# 결측치 처리
X_t = pd.DataFrame(columns=['지점', '지점명', '일시', '기온(°C)', '풍속(m/s)', '풍향(16방위)', '습도(%)', '증기압(hPa)','이슬점온도(°C)', '현지기압(hPa)', '해면기압(hPa)', '전운량(10분위)', '시정(10m)','지면온도(°C)'])

# 각 지역별로 처리
for i in range(len(지점_list)):
  X_t = pd.concat([X_t,temp2023_02.query(f'지점=={지점_list[i]}').interpolate()])

# '일시'컬럼 datetime형식으로 변환
X_t['일시'] = pd.to_datetime(X_t['일시'])

# 인덱스 추출
X_t_index = X_t.iloc[:,:3]

# Test데이터 생성
X_t = X_t.iloc[:,3:]

#### Scaling

In [ ]:
# Scaling
X[num_features] = scaler.fit_transform(X[num_features])
X_t[num_features] = scaler.transform(X_t[num_features])

#### LGBMClassifier

In [ ]:
# Define objective function for Optuna
def objective_lgbm(trial):
    # Define parameters to be optimized
    lgbm_params = {
        'objective': 'binary',
        'boosting_type': 'gbdt',
        'verbosity': -1,
        'random_state' : CFG.SEED,
        'num_leaves': trial.suggest_int('num_leaves', 2, 256),
        'learning_rate': trial.suggest_uniform('learning_rate', 0.01, 0.1),
        'feature_fraction': trial.suggest_uniform('feature_fraction', 0.1, 1.0),
        'bagging_fraction': trial.suggest_uniform('bagging_fraction', 0.1, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
    }
    
    # Create LGBM classifier with parameters
    lgbm_clf = lgb.LGBMClassifier(**lgbm_params)
    
    # Compute cross validation score
    lgbm_score = cross_val_score(lgbm_clf, X, y, cv=15).mean()
    
    return lgbm_score

# Run optuna to optimize hyperparameters
study_lgbm = optuna.create_study(direction='maximize')
study_lgbm.optimize(objective_lgbm, n_trials=15)

# Get best hyperparameters and create LGBM classifier with those parameters
lgbm_params = study_lgbm.best_params
lgbm_model = lgb.LGBMClassifier(**lgbm_params)

#### CatBoostClassifier

In [ ]:
# Define objective function for Optuna for CatBoost
def objective_cb(trial):
    # Define parameters to be optimized
    cb_params = {
        'loss_function': 'Logloss',
        'iterations': 1000,
        'task_type' : "GPU",
        'verbose' : False,
        'random_state' : CFG.SEED,
        'learning_rate': trial.suggest_uniform('learning_rate', 0.01, 0.1),
        'depth': trial.suggest_int('depth', 2, 10),
        'l2_leaf_reg': trial.suggest_loguniform('l2_leaf_reg', 0.01, 10.0),
        'random_strength': trial.suggest_uniform('random_strength', 0.1, 1.0),
        'bagging_temperature': trial.suggest_uniform('bagging_temperature', 0.0, 10.0),
        'border_count': trial.suggest_int('border_count', 1, 255),
    }

    # Create Catboost classifier with parameters
    cb_clf = cb.CatBoostClassifier(**cb_params)
    
    # Compute cross validation score
    cb_score = cross_val_score(cb_clf, X, y, cv=5).mean()
    
    return cb_score    

# Run optuna to optimize hyperparameters
study_cb = optuna.create_study(direction='maximize')
study_cb.optimize(objective_cb, n_trials=5)

# Get best hyperparameters and create Catboost classifier with those parameters
cb_params = study_cb.best_params
catboost_model = cb.CatBoostClassifier(**cb_params)

#### Ensemble

In [ ]:
# Create voting ensemble of LightGBM and CatBoost models
ensemble_model = VotingClassifier(estimators=[('lgbm', lgbm_model), ('catboost', catboost_model)], voting='soft')
ensemble_model.fit(X, y)

#### Model Save

In [ ]:
# 학습된 모델 저장
with open('traffic_accident_prediction.pkl', 'wb') as f:
    pickle.dump(ensemble_model, f)

### 결과

In [ ]:
#생성한 예측 모델로 실제 미래데이터를 예측하기
y_t_pred = ensemble_model.predict(X_t)
y_t_pred_proba = ensemble_model.predict_proba(X_t)

In [ ]:
X_t_y_p = X_t_index.copy()
X_t_y_proba = X_t_index.copy()
X_t_y_p['예측사고유무'] = y_t_pred
X_t_y_proba['예측사고확률'] = [row[1] for row in y_t_pred_proba]
X_t_y_proba['예측사고확률'] = X_t_y_proba['예측사고확률'].apply(lambda x: str(round(x * 100, 2)) + '%')

In [ ]:
X_t_y_p

In [ ]:
X_t_y_proba

## 지역통합
- 실시간 일기예보 같은 경우는 종관기상관측(ASOS)를 사용한다. 가장 정확성이 높은 ASOS를 기반으로 인근 지역 통합 일기 예보를 하기 때문에 ASOS 관측 데이터를 사용하였다.
- ASOS는 충남,대전,세종에 대표적으로 ('서산', '대전', '홍성', '천안', '보령', '부여', '금산', '세종')이렇게 8군데가 있는데 교통사고 데이터와 접목하기 위해서 8군데 위도,경도와 실제 교통사고데이터 위도,경도를 이용하여 제일 가까운 ASOS 관측소 날씨데이터를 이용하였다.
- 교통사고 데이터는 충청남도 전체에서 발생했기 때문에 예를들어 모델에서 '서산'이라고 예측을 했어도 서산 뿐만 아니라 태안, 당진일 가능성도 있기 때문에 ASOS 관측소 위치 기반으로 지역을 묶어보였다. 

In [ ]:
X_index.지점명.unique()

![이미지](https://www.kma.go.kr/daejeon/images/info/area2_2.jpg)

[이미지출처] : https://www.kma.go.kr/daejeon/html/info/business01.jsp

- 지점명에['서산', '대전', '홍성', '천안', '보령', '부여', '금산', '세종']와 같이 8가지 지역밖에 없는데 일기예보가 실시간으로 예보하는 대표적인 8개지역이다.
-기상청 관측소 위도,경도를 이용해 사고시 가장 가까운 관측소 데이터를 활용하였는데 모델에서 나온 지역명을 이렇게 해석하면 된다.
- 서산 -> 서산,당진,태안
- 대전 -> 대전,계룡
- 홍성 -> 홍성,예산
- 천안 -> 천안,아산
- 보령 -> 보령,서천
- 부여 -> 부여,청양
- 금산 -> 금산
- 세종 -> 세종,공주

# 추후 발전가능성 및 실용성

- **실시간 교통 사고 예측 모델로 사고 발생 예측 확률이 높은 지역을 우선적으로 순찰하여 교통사고 방지 및 빠른 대처가 가능해진다.**

- **실시간 교통 사고 예측 모델은 실시간으로 기상청 Open Api를 이용하여 1시간 이내 각 지역별 사고 예측 확률 및 사고 발생 여부를 추가적으로 접목시킬 수 있다.**

- **실시간 교통 사고 예측 모델은 시계열 데이터 예측 모델이기 때문에 시간이 지날 수록 더 많은 데이터를 학습이 가능하므로 성능이 계속 올라 갈 수 있다.**

- **실시간 교통 사고 예측 모델은 시간 단위 데이터를 활용하였지만 기상청 일단위 데이터를 사용한다면 같은 매커니즘으로 일단위 사고 예측 확률 및 사고예측 여부 모델 개발이 가능하다.**